# 🧬 Image Data Augmentation Research Lab

## Google Colab Research Companion

**Module:** CO3091 / CO7091 — Computational Intelligence and Software Engineering  
**Source paper:** Shorten, C. & Khoshgoftaar, T. M. (2019), *A survey on Image Data Augmentation for Deep Learning*, Journal of Big Data, 6, 60.

### What this notebook does

This notebook turns the repository into a complete, traceable experiment:

**Upload → inspect → augment → compare → measure → repeat → export → interpret**

It demonstrates image-space augmentation discussed in the survey and keeps **literature evidence**, **reproduction**, and **new experimental extensions** separate.

> **Important:** Pixel-level metrics measure image change. They do not prove that an augmentation improves CNN accuracy or generalisation.


In [ ]:
!pip -q install pillow numpy pandas matplotlib


## 1. Environment and reproducibility

The first cell imports the experimental dependencies and fixes a global seed for repeatable demonstrations.


In [ ]:
from google.colab import files
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from IPython.display import display
import io, json, hashlib, zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

GLOBAL_SEED = 2026
rng = np.random.default_rng(GLOBAL_SEED)

print("NumPy:", np.__version__)
print("Global experiment seed:", GLOBAL_SEED)


## 2. Upload the source image

Upload one JPG, JPEG or PNG image.

The notebook converts it to RGB so every augmentation and diagnostic uses the same three-channel representation.


In [ ]:
uploaded = files.upload()

if not uploaded:
    raise ValueError("No image was uploaded.")

filename = next(iter(uploaded))
original = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")

print(f"Loaded file: {filename}")
print(f"Image size: {original.size[0]} × {original.size[1]} pixels")
print(f"Mode: {original.mode}")

display(original)


## 3. Image inspection

Before transforming the image, record its basic properties. This is useful when experiments need to be reproduced later.


In [ ]:
def image_summary(image, name="image"):
    arr = np.asarray(image)
    return {
        "name": name,
        "width": image.width,
        "height": image.height,
        "channels": arr.shape[-1],
        "mean_R": float(arr[:,:,0].mean()),
        "mean_G": float(arr[:,:,1].mean()),
        "mean_B": float(arr[:,:,2].mean()),
        "std_R": float(arr[:,:,0].std()),
        "std_G": float(arr[:,:,1].std()),
        "std_B": float(arr[:,:,2].std()),
    }

pd.DataFrame([image_summary(original, filename)])


## 4. Survey taxonomy used in this notebook

The survey discusses a broad augmentation landscape. The executable image-space portion of this notebook focuses on:

1. **Geometric transformations** — modify spatial structure.
2. **Colour-space transformations** — modify appearance or colour statistics.
3. **Kernel filters** — modify local image structure.
4. **Occlusion / random erasing** — remove or replace regions.
5. **Image mixing** — combine images.

The survey also discusses feature-space augmentation, adversarial training, GAN-based augmentation, neural style transfer and meta-learning. Those methods require substantially different model-level implementations and are therefore documented rather than falsely claimed as reproduced here.


# 5. Geometric augmentation

### What happens?

Geometric operations change the spatial arrangement of pixels. Examples include flipping, rotation, translation, shear and crop-and-resize.

**Research question:** Does a transformation preserve the semantic label for the target dataset?

That question cannot be answered from pixels alone; it depends on the task and dataset.


In [ ]:
def rotate(image, degrees):
    return image.rotate(
        degrees,
        expand=True,
        fillcolor=(255, 255, 255),
        resample=Image.Resampling.BICUBIC
    )

def translate(image, x, y):
    return image.transform(
        image.size,
        Image.Transform.AFFINE,
        (1, 0, -x, 0, 1, -y),
        resample=Image.Resampling.BICUBIC,
        fillcolor=(255, 255, 255)
    )

def shear(image, x_shear, y_shear):
    return image.transform(
        image.size,
        Image.Transform.AFFINE,
        (1, x_shear, 0, y_shear, 1, 0),
        resample=Image.Resampling.BICUBIC,
        fillcolor=(255, 255, 255)
    )

def crop_resize(image, fraction=0.8):
    w, h = image.size
    nw, nh = int(w*fraction), int(h*fraction)
    left, top = (w-nw)//2, (h-nh)//2
    return image.crop((left, top, left+nw, top+nh)).resize(
        (w, h), Image.Resampling.LANCZOS
    )

geometric = {
    "Original": original,
    "Horizontal flip": original.transpose(Image.Transpose.FLIP_LEFT_RIGHT),
    "Vertical flip": original.transpose(Image.Transpose.FLIP_TOP_BOTTOM),
    "Rotation +25°": rotate(original, 25),
    "Translation": translate(original, 25, 20),
    "Shear": shear(original, 0.15, 0),
    "Crop + resize": crop_resize(original, 0.8),
}

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, (title, image) in zip(axes.ravel(), geometric.items()):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


# 6. Colour-space augmentation

### What happens?

These operations alter appearance while attempting to preserve the underlying object or scene.

The examples below change brightness, contrast, saturation, sharpness and grayscale representation.


In [ ]:
brightness = ImageEnhance.Brightness(original).enhance(1.3)
contrast = ImageEnhance.Contrast(original).enhance(1.3)
saturation = ImageEnhance.Color(original).enhance(0.7)
sharpness = ImageEnhance.Sharpness(original).enhance(1.6)
grayscale = ImageOps.grayscale(original).convert("RGB")
inverted = ImageOps.invert(original)

colour_examples = {
    "Original": original,
    "Brightness ×1.3": brightness,
    "Contrast ×1.3": contrast,
    "Saturation ×0.7": saturation,
    "Sharpness ×1.6": sharpness,
    "Grayscale": grayscale,
    "Invert": inverted,
}

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, (title, image) in zip(axes.ravel(), colour_examples.items()):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


# 7. Kernel filters

### What happens?

Kernel and filtering operations alter local image structure. They can blur, sharpen, enhance edges or create stylised local responses.

These transformations can produce useful variation, but aggressive filtering may also create unrealistic training examples.


In [ ]:
kernel_examples = {
    "Original": original,
    "Blur": original.filter(ImageFilter.BLUR),
    "Gaussian blur": original.filter(ImageFilter.GaussianBlur(2.0)),
    "Sharpen": original.filter(ImageFilter.SHARPEN),
    "Edge enhance": original.filter(ImageFilter.EDGE_ENHANCE),
    "Emboss": original.filter(ImageFilter.EMBOSS),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (title, image) in zip(axes.ravel(), kernel_examples.items()):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


# 8. Occlusion: random erasing

### What happens?

Random erasing removes a rectangular region. The location and shape are sampled using a random generator.

This creates partial occlusion and can reduce reliance on a single visible region, but the correct label-preservation behaviour is dataset dependent.


In [ ]:
def random_erasing(image, area_percent=10, seed=2026, fill_mode="black"):
    local_rng = np.random.default_rng(seed)
    array = np.asarray(image).copy()
    height, width, _ = array.shape

    target = max(1, int(height * width * area_percent / 100))
    ratio = float(local_rng.uniform(0.5, 2.0))

    erase_h = max(1, min(height, int(np.sqrt(target * ratio))))
    erase_w = max(1, min(width, int(np.sqrt(target / ratio))))

    y = int(local_rng.integers(0, max(1, height - erase_h + 1)))
    x = int(local_rng.integers(0, max(1, width - erase_w + 1)))

    if fill_mode == "random":
        array[y:y+erase_h, x:x+erase_w] = local_rng.integers(
            0, 256, size=(erase_h, erase_w, 3), dtype=np.uint8
        )
    elif fill_mode == "mean":
        array[y:y+erase_h, x:x+erase_w] = np.mean(
            array, axis=(0,1)
        ).astype(np.uint8)
    else:
        array[y:y+erase_h, x:x+erase_w] = 0

    return Image.fromarray(array)

erased = random_erasing(original, area_percent=10, seed=GLOBAL_SEED)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(original); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(erased); axes[1].set_title("Random erasing — 10%"); axes[1].axis("off")
plt.tight_layout()
plt.show()


# 9. Image mixing

### What happens?

Image mixing creates a new sample from two source images.

This notebook uses a simple weighted pixel interpolation as a **demonstration of the mixing idea**. It is not a claim to reproduce every mixing method discussed in the literature.


In [ ]:
second_upload = files.upload()

if second_upload:
    second_name = next(iter(second_upload))
    second_image = Image.open(io.BytesIO(second_upload[second_name])).convert("RGB")
    second_image = second_image.resize(original.size, Image.Resampling.LANCZOS)

    alpha = 0.5
    mixed = Image.blend(original, second_image, alpha)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].imshow(original); axes[0].set_title("Image A"); axes[0].axis("off")
    axes[1].imshow(second_image); axes[1].set_title("Image B"); axes[1].axis("off")
    axes[2].imshow(mixed); axes[2].set_title("50/50 mixed image"); axes[2].axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No second image supplied — mixing demonstration skipped.")


# 10. Quantitative image-level diagnostics

The notebook now measures how strongly an augmentation changes the source image.

**MSE:** average squared pixel difference.  
**MAE:** average absolute pixel difference.  
**PSNR:** logarithmic similarity measure derived from MSE.  
**Histogram distance:** difference between RGB intensity distributions.  
**Edge density:** simple local-gradient measure.

These are **diagnostic measurements**, not classification metrics.


In [ ]:
def resized_array(image, size=(256, 256)):
    return np.asarray(image.resize(size), dtype=np.float32)

def mse(a, b):
    x, y = resized_array(a), resized_array(b)
    return float(np.mean((x-y)**2))

def mae(a, b):
    x, y = resized_array(a), resized_array(b)
    return float(np.mean(np.abs(x-y)))

def psnr(a, b):
    value = mse(a, b)
    return float("inf") if value == 0 else float(10*np.log10((255.0**2)/value))

def rgb_histogram(image, bins=32):
    arr = resized_array(image)
    histograms = []
    for channel in range(3):
        h, _ = np.histogram(
            arr[:,:,channel],
            bins=bins,
            range=(0,256),
            density=True
        )
        histograms.append(h)
    return np.concatenate(histograms)

def histogram_distance(a, b):
    ha, hb = rgb_histogram(a), rgb_histogram(b)
    denominator = np.linalg.norm(ha) * np.linalg.norm(hb)
    return 0.0 if denominator == 0 else float(
        1 - np.dot(ha, hb) / denominator
    )

def edge_density(image):
    arr = resized_array(image)
    gray = 0.299*arr[:,:,0] + 0.587*arr[:,:,1] + 0.114*arr[:,:,2]
    gx = np.abs(np.diff(gray, axis=1))
    gy = np.abs(np.diff(gray, axis=0))
    return float((gx.mean()+gy.mean())/(2*255))

def diagnostics(original, augmented):
    original_edge = edge_density(original)
    augmented_edge = edge_density(augmented)
    return {
        "MSE": mse(original, augmented),
        "MAE": mae(original, augmented),
        "PSNR_dB": psnr(original, augmented),
        "Histogram_distance": histogram_distance(original, augmented),
        "Edge_density_original": original_edge,
        "Edge_density_augmented": augmented_edge,
        "Edge_density_delta": augmented_edge-original_edge,
    }

diagnostics(original, erased)


## 11. Compare the augmentation families

This section creates a reproducible table across the transformations demonstrated above.


In [ ]:
comparison_images = {
    "Horizontal flip": geometric["Horizontal flip"],
    "Rotation": geometric["Rotation +25°"],
    "Brightness": brightness,
    "Contrast": contrast,
    "Saturation": saturation,
    "Gaussian blur": kernel_examples["Gaussian blur"],
    "Random erasing": erased,
}

rows = []
for name, image in comparison_images.items():
    rows.append({"augmentation": name, **diagnostics(original, image)})

comparison_df = pd.DataFrame(rows)
display(comparison_df.round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(comparison_df["augmentation"], comparison_df["MAE"])
axes[0].set_title("Mean Absolute Error")
axes[0].set_ylabel("MAE")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(comparison_df["augmentation"], comparison_df["Histogram_distance"])
axes[1].set_title("RGB Histogram Distance")
axes[1].set_ylabel("Distance")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


# 12. Stochastic augmentation policy

A research experiment often needs more than one fixed transformation. This section creates a policy containing operations, probabilities and magnitudes.

The policy is an **experimental extension** of the repository. It is not presented as a reproduction of AutoAugment.


In [ ]:
from dataclasses import dataclass, asdict

@dataclass
class PolicyStep:
    operation: str
    probability: float
    magnitude: float

def apply_policy_step(image, step, local_rng):
    if local_rng.random() > step.probability:
        return image, False

    m = step.magnitude
    op = step.operation

    if op == "Horizontal flip":
        return image.transpose(Image.Transpose.FLIP_LEFT_RIGHT), True
    if op == "Rotation":
        return rotate(image, float(local_rng.uniform(-30,30)*m)), True
    if op == "Brightness":
        return ImageEnhance.Brightness(image).enhance(
            float(1+local_rng.uniform(-0.4,0.4)*m)
        ), True
    if op == "Contrast":
        return ImageEnhance.Contrast(image).enhance(
            float(1+local_rng.uniform(-0.4,0.4)*m)
        ), True
    if op == "Saturation":
        return ImageEnhance.Color(image).enhance(
            float(1+local_rng.uniform(-0.5,0.5)*m)
        ), True
    if op == "Gaussian blur":
        return image.filter(
            ImageFilter.GaussianBlur(float(max(0.05, local_rng.uniform(0.2,4.0)*m)))
        ), True
    if op == "Random erasing":
        return random_erasing(
            image,
            area_percent=float(local_rng.uniform(5,25)*m),
            seed=int(local_rng.integers(0,10_000_000)),
            fill_mode="random"
        ), True

    return image, True

policy = [
    PolicyStep("Horizontal flip", 0.7, 1.0),
    PolicyStep("Rotation", 0.8, 0.7),
    PolicyStep("Brightness", 0.6, 0.5),
]

policy_rng = np.random.default_rng(GLOBAL_SEED)
policy_output = original.copy()
policy_trace = []

for step in policy:
    policy_output, applied = apply_policy_step(policy_output, step, policy_rng)
    policy_trace.append({
        **asdict(step),
        "applied": applied
    })

display(pd.DataFrame(policy_trace))
display(policy_output)


# 13. Repeated experiment matrix

One transformation on one image is not enough to establish a general conclusion.

This section repeats the same controlled transformations with multiple seeds and summarises the resulting image-level measurements.


In [ ]:
EXPERIMENTS = {
    "Rotation": lambda seed: rotate(original, 30),
    "Brightness": lambda seed: ImageEnhance.Brightness(original).enhance(1.25),
    "Contrast": lambda seed: ImageEnhance.Contrast(original).enhance(1.25),
    "Saturation": lambda seed: ImageEnhance.Color(original).enhance(1.25),
    "Gaussian blur": lambda seed: original.filter(ImageFilter.GaussianBlur(2.0)),
    "Random erasing": lambda seed: random_erasing(
        original, 10, seed=seed, fill_mode="random"
    ),
}

REPETITIONS = 5
matrix_rows = []

for operation, function in EXPERIMENTS.items():
    for repetition in range(REPETITIONS):
        seed = int(GLOBAL_SEED + repetition)
        output = function(seed)
        matrix_rows.append({
            "operation": operation,
            "repetition": repetition+1,
            "seed": seed,
            **diagnostics(original, output)
        })

matrix_df = pd.DataFrame(matrix_rows)

summary_df = (
    matrix_df.groupby("operation")
    .agg(
        MAE_mean=("MAE","mean"),
        MAE_std=("MAE","std"),
        PSNR_mean=("PSNR_dB","mean"),
        Histogram_distance_mean=("Histogram_distance","mean"),
        Edge_delta_mean=("Edge_density_delta","mean"),
    )
    .reset_index()
    .fillna(0)
)

display(summary_df.round(4))


In [ ]:
plt.figure(figsize=(12,5))
plt.bar(summary_df["operation"], summary_df["MAE_mean"])
plt.ylabel("Mean MAE")
plt.title("Repeated experiment: mean image change")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# 14. Dataset generator

The following section generates multiple augmented variants and records a manifest.

The manifest is important because an experiment should retain the relationship between:

**source image → variant → random seed → augmentation trace**


In [ ]:
dataset_rng = np.random.default_rng(GLOBAL_SEED)
generated = []
manifest = []

policy_choices = [
    "Horizontal flip",
    "Rotation",
    "Brightness",
    "Contrast",
    "Saturation",
    "Gaussian blur",
    "Random erasing",
]

for variant_id in range(10):
    seed = int(dataset_rng.integers(0,10_000_000))
    op = str(dataset_rng.choice(policy_choices))

    if op == "Horizontal flip":
        output = original.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    elif op == "Rotation":
        output = rotate(original, float(dataset_rng.uniform(-30,30)))
    elif op == "Brightness":
        output = ImageEnhance.Brightness(original).enhance(
            float(dataset_rng.uniform(0.7,1.3))
        )
    elif op == "Contrast":
        output = ImageEnhance.Contrast(original).enhance(
            float(dataset_rng.uniform(0.7,1.3))
        )
    elif op == "Saturation":
        output = ImageEnhance.Color(original).enhance(
            float(dataset_rng.uniform(0.6,1.4))
        )
    elif op == "Gaussian blur":
        output = original.filter(
            ImageFilter.GaussianBlur(float(dataset_rng.uniform(0.5,3.0)))
        )
    else:
        output = random_erasing(
            original,
            area_percent=float(dataset_rng.uniform(5,20)),
            seed=seed,
            fill_mode="random"
        )

    generated.append(output)

    manifest.append({
        "variant_id": variant_id,
        "source": filename,
        "operation": op,
        "seed": seed,
    })

manifest_df = pd.DataFrame(manifest)
display(manifest_df)

fig, axes = plt.subplots(2,5,figsize=(18,8))
for ax, image, row in zip(axes.ravel(), generated, manifest):
    ax.imshow(image)
    ax.set_title(f"#{row['variant_id']} {row['operation']}")
    ax.axis("off")
plt.tight_layout()
plt.show()


# 15. Export experiment results

The notebook exports both measurements and provenance metadata so the experiment can be inspected outside Colab.


In [ ]:
experiment_id = hashlib.sha256(
    matrix_df.to_csv(index=False).encode("utf-8")
).hexdigest()[:12]

metadata = {
    "experiment_id": experiment_id,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_file": filename,
    "source_size": list(original.size),
    "global_seed": GLOBAL_SEED,
    "repetitions": REPETITIONS,
    "scope": "image-level diagnostics",
    "model_accuracy_claim": False,
}

matrix_df.to_csv("experiment_matrix.csv", index=False)
summary_df.to_csv("experiment_summary.csv", index=False)
manifest_df.to_csv("augmentation_manifest.csv", index=False)

with open("experiment_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Experiment ID:", experiment_id)
print(json.dumps(metadata, indent=2))

files.download("experiment_matrix.csv")
files.download("experiment_summary.csv")
files.download("augmentation_manifest.csv")
files.download("experiment_metadata.json")


# 16. Research interpretation

### What this notebook establishes

It demonstrates how image-space augmentation can be:

- implemented;
- visualised;
- quantified;
- repeated under controlled seeds;
- compared using image-level diagnostics;
- exported with experiment metadata.

### What it does not establish

It does **not** establish that:

- one augmentation is universally better;
- an image remains correctly labelled after every transformation;
- a CNN will achieve higher accuracy;
- the generated images improve generalisation;
- the experiments reproduce all numerical findings from the survey.

A model-level study would require a fixed dataset, fixed architecture, training-only augmentation, isolated validation/test data, repeated runs and task-specific metrics.

---

## 17. Relation to the source paper

The repository implements a substantial subset of the survey's image-space techniques.

The following are intentionally **not claimed as reproduced** in this notebook:

- feature-space augmentation;
- adversarial training;
- GAN-based augmentation;
- neural style transfer;
- the complete meta-learning literature.

Those methods require model-level experimental designs beyond this image-space notebook.

### Reference

Shorten, C., & Khoshgoftaar, T. M. (2019). *A survey on Image Data Augmentation for Deep Learning*. **Journal of Big Data, 6**, 60.

DOI: **10.1186/s40537-019-0197-0**


## ✅ End-to-end experiment checklist

- [x] Environment configured
- [x] Source image uploaded
- [x] Image properties recorded
- [x] Geometric augmentation
- [x] Colour-space augmentation
- [x] Kernel filters
- [x] Random erasing
- [x] Image mixing demonstration
- [x] Pixel-level diagnostics
- [x] Cross-technique comparison
- [x] Stochastic policy
- [x] Repeated experiment matrix
- [x] Dataset generation
- [x] CSV/JSON export
- [x] Research interpretation and limitations

**Next research level:** connect the generated training data to a fixed CNN benchmark and evaluate validation/test performance with repeated seeds. That would be a new model-level experiment, not a claim that the survey's experiments have been reproduced.
